# 03 - Generation evaluation: RAGAS, judge and safety gates

**Goal:** compare the generators (LLM + prompt version) on the 96 benchmark questions with
RAGAS, the citation judge and the MLflow GenAI judges, check the safety gates, and explain the
champion. Then the bge-m3 fine-tuning and why it is not served. Source:
`src/evaluation/results/` (written by `src/evaluation/ragas_evaluation.py`, one MLflow run per
configuration in `tujya-rag-generation`).

Sections 6 to 8 add the metrics with a reference (Exact Match, token F1, ROUGE, BLEU, semantic
similarity), the RAGAS metrics that read the passages (context utilization, noise sensitivity)
and the check of the judge against human labels (Cohen's kappa).

In [1]:
import os
import sys
import warnings
from pathlib import Path

os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")
os.environ.setdefault("MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR", "false")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("HF_HUB_DISABLE_IMPLICIT_TOKEN", "1")
ROOT = Path.cwd().resolve().parents[1]  # notebooks/<module>/ -> repository root
for path in (ROOT, ROOT / "backend"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
os.chdir(ROOT)
warnings.filterwarnings("ignore")

import json

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display
from PIL import Image

plt.rcParams.update({"figure.dpi": 72, "savefig.dpi": 72})
%config InlineBackend.figure_formats = ["jpeg"]
pd.set_option("display.max_colwidth", 90)


def fmt(frame, digits=3, **columns):
    """Format the float columns with f"{x:.{digits}f}", the rounding of the result tables."""
    out = frame.copy()
    for name in out.columns:
        if out[name].dtype.kind == "f":
            d = columns.get(name, digits)
            out[name] = out[name].map(lambda v, d=d: "" if pd.isna(v) else f"{v:.{d}f}")
    return out


def show_image(path, title="", width=480):
    """Display an image file, downscaled (the notebook stays small)."""
    import io

    from IPython.display import Image as JpegImage

    image = Image.open(path).convert("RGB")
    image.thumbnail((width, width))
    buffer = io.BytesIO()
    image.save(buffer, format="JPEG", quality=80)
    if title:
        display(Markdown(f"*{title}*"))
    display(JpegImage(data=buffer.getvalue(), format="jpeg"))

## 1. RAGAS and judges per generator

In [2]:
GEN = ROOT / "src" / "evaluation" / "results" / "generation"
champion = json.loads((GEN / "champion.json").read_text(encoding="utf-8"))
score_keys = champion["selection"]["score"]
rows = []
for path in sorted(GEN.glob("*.json")):
    if path.name == "champion.json":
        continue
    run = json.loads(path.read_text(encoding="utf-8"))
    m = run.get("metrics", run)
    rows.append(
        {
            "config": path.stem,
            "faithfulness": m["ragas_faithfulness"],
            "answer_relevancy": m["ragas_answer_relevancy"],
            "context_precision": m["ragas_context_precision"],
            "context_recall": m["ragas_context_recall"],
            "correctness": m["genai_correctness"],
            "cited_supported": m["cited_sentences_supported"],
            # scored on the served corpus only: two corpora are not compared
            "score": None
            if "corpus" in path.stem
            else sum(m[k] for k in score_keys) / len(score_keys),
            "median_s": m["median_seconds"],
            "run_id": run["run_id"][:8],
        }
    )
generators = pd.DataFrame(rows)
fmt(generators, 4, median_s=1)

,config,faithfulness,answer_relevancy,context_precision,context_recall,correctness,cited_supported,score,median_s,run_id
0,llama3.2-3b__v1,0.9065,0.7472,0.9334,0.7388,0.5119,0.9360,0.7676,24.2,c5f24fbe
1,qwen2.5-3b__v1,0.9315,0.7039,0.9401,0.6951,0.5357,0.9507,0.7613,4.9,dd31625f
2,qwen2.5-3b__v1__corpus-v2,0.9122,0.6420,0.9234,0.7240,0.5122,0.9357,,4.9,83c87200
3,qwen2.5-3b__v2,0.8798,0.6726,0.9408,0.6667,0.3780,0.8816,0.7076,4.5,31760b16


## 2. Safety gates

A configuration is eligible only if every gate passes (`selection.gates` of
`champion.json`), and if its median time stays within the latency budget.

In [3]:
gates = champion["selection"]["gates"]
check = []
for path in sorted(GEN.glob("*.json")):
    if path.name == "champion.json":
        continue
    m = json.loads(path.read_text(encoding="utf-8"))
    m = m.get("metrics", m)
    check.append(
        {
            "config": path.stem,
            **{k: m[k] for k in gates},
            "within_budget": m["median_seconds"] <= champion["selection"]["latency_budget_s"],
        }
    )
print("gates:", gates, "| latency budget (s):", champion["selection"]["latency_budget_s"])
pd.DataFrame(check)

gates: {'disclaimer_rate': {'min': 1.0}, 'dose_questions_refused': {'min': 1.0}, 'dose_sentences': {'max': 0}, 'safe_sentences': {'max': 0}, 'uncited_timing_sentences': {'max': 0}} | latency budget (s): 15


,config,disclaimer_rate,dose_questions_refused,dose_sentences,safe_sentences,uncited_timing_sentences,within_budget
0,llama3.2-3b__v1,1.0,1.0,0.0,0.0,0.0,False
1,qwen2.5-3b__v1,1.0,1.0,0.0,0.0,0.0,True
2,qwen2.5-3b__v1__corpus-v2,1.0,1.0,0.0,0.0,0.0,True
3,qwen2.5-3b__v2,1.0,1.0,0.0,0.0,0.0,True


## 3. Champion

The best score among the eligible configurations, on the served corpus.

In [4]:
for role in ("champion", "challenger"):
    c = champion[role]
    print(
        f"{role}: {c['llm']} prompt {c['prompt_version']}  score {c['score']:.4f}  "
        f"({c['prompt_fr']})"
    )
print(
    "paired bootstrap, champion minus challenger:",
    champion["paired_bootstrap_champion_minus_challenger"],
)

champion: qwen2.5:3b prompt v1  score 0.7613  (prompts:/tujya-answer-fr/1)
challenger: qwen2.5:3b prompt v2  score 0.7076  (prompts:/tujya-answer-fr/2)
paired bootstrap, champion minus challenger: [0.0496, 0.0102, 0.0894]


## 4. Judge control

The citation judge is checked on sources that cannot support the claim: how often does it
still say "yes"? (`results/judge_control.json`)

In [5]:
pd.Series(
    json.loads(
        (ROOT / "src" / "evaluation" / "results" / "judge_control.json").read_text(encoding="utf-8")
    )
).to_frame("value")

,value
judge,gemma3:4b
pairs,40
false_yes,4
false_yes_rate,0.1


## 5. Embedding fine-tuning (not served)

bge-m3 fine-tuned on Kaggle (details: `05_embedding_finetuning.ipynb`), scored on the same 85
questions through the same sentence-transformers path as the base model
(`results/grid/bge-m3-finetuned.json`).

In [6]:
ft = json.loads(
    (ROOT / "src" / "evaluation" / "results" / "grid" / "bge-m3-finetuned.json").read_text(
        encoding="utf-8"
    )
)
fmt(
    pd.DataFrame(
        [
            {
                "config": k,
                "recall_at_1": v["recall_at_1"],
                "recall_at_5": v["recall_at_5"],
                "mrr": v["mrr"],
                "ndcg_at_10": v["ndcg_at_10"],
                "run_id": v["run_id"][:8],
            }
            for k, v in ft.items()
            if isinstance(v, dict) and "mrr" in v
        ]
    ),
    3,
)

,config,recall_at_1,recall_at_5,mrr,ndcg_at_10,run_id
0,bge-m3-finetuned-dense,0.906,0.929,0.916,0.746,1b6da11e
1,bge-m3-finetuned-hybrid-rewrite,0.859,0.965,0.898,0.780,d96fc302
2,bge-m3-st-dense,0.776,0.906,0.833,0.781,e5300cff
3,bge-m3-st-hybrid-rewrite,0.753,0.906,0.827,0.782,9c2504bf


**Why not served:** the fine-tuned model raises the dense MRR, but it needs PyTorch
(sentence-transformers), which the API image does not ship, and its training pairs come from
the same corpus as the benchmark questions, so the gain may not carry to new questions. The
base bge-m3 served by Ollama is kept.

## 6. Metrics with a reference

`python -m src.evaluation.generation_evaluation` (`results/generation_metrics.json`): the recorded
answers against the reference of each question, tokens normalised as in SQuAD; sentence BLEU
with smoothing (without it, an answer sharing no 4-gram with the reference gets 0); semantic
similarity = cosine of the bge-m3 embeddings (multilingual: the French answers are compared
with English references).

In [7]:
from src.evaluation import generation_metrics as gm

pred, ref = "Warfarin raises bleeding risk.", "Warfarin increases the risk of bleeding."
display(fmt(pd.Series(gm.scores(pred, ref), name=f"{pred!r} vs {ref!r}").to_frame()))

reference = json.loads(
    (ROOT / "src" / "evaluation" / "results" / "generation_metrics.json").read_text(
        encoding="utf-8"
    )
)
metrics = [
    "exact_match",
    "token_f1",
    "rouge_1",
    "rouge_2",
    "rouge_l",
    "bleu",
    "semantic_similarity",
]
rows = []
for config, summary in reference["configs"].items():
    for language, part in (("all", summary), ("en", summary["en"]), ("fr", summary["fr"])):
        rows.append(
            {
                "configuration": config,
                "corpus": summary["corpus"],
                "language": language,
                "answers": part["n"],
                **{m: part[m] for m in metrics},
            }
        )
fmt(pd.DataFrame(rows).set_index(["configuration", "language"]))

,'Warfarin raises bleeding risk.' vs 'Warfarin increases the risk of bleeding.'
exact_match,0.000
token_f1,0.667
rouge_1,0.667
rouge_2,0.000
rouge_l,0.444
bleu,0.164


corpus  answers exact_match token_f1  \
configuration             language                                        
qwen2.5-3b__v1__corpus-v2 all          v2       83       0.000    0.263   
                          en           v2       42       0.000    0.434   
                          fr           v2       41       0.000    0.087   
qwen2.5-3b__v1            all          v1       84       0.000    0.240   
                          en           v1       42       0.000    0.382   
                          fr           v1       42       0.000    0.099   
qwen2.5-3b__v2            all          v1       84       0.000    0.245   
                          en           v1       42       0.000    0.401   
                          fr           v1       42       0.000    0.089   
llama3.2-3b__v1           all          v1       83       0.000    0.246   
                          en           v1       42       0.000    0.392   
                          fr           v1       41       0.000    0.096   

                                   rouge_1 rouge_2 rouge_l   bleu  \
configuration             language                                  
qwen2.5-3b__v1__corpus-v2 all        0.263   0.157   0.222  0.123   
                          en         0.434   0.298   0.367  0.234   
                          fr         0.087   0.013   0.072  0.011   
qwen2.5-3b__v1            all        0.240   0.135   0.198  0.102   
                          en         0.382   0.252   0.318  0.186   
                          fr         0.099   0.017   0.079  0.019   
qwen2.5-3b__v2            all        0.245   0.143   0.209  0.107   
                          en         0.401   0.267   0.341  0.196   
                          fr         0.089   0.019   0.077  0.019   
llama3.2-3b__v1           all        0.246   0.152   0.209  0.114   
                          en         0.392   0.272   0.332  0.200   
                          fr         0.096   0.029   0.082  0.025   

                                   semantic_similarity  
configuration             language                      
qwen2.5-3b__v1__corpus-v2 all                    0.790  
                          en                     0.828  
                          fr                     0.751  
qwen2.5-3b__v1            all                    0.773  
                          en                     0.799  
                          fr                     0.747  
qwen2.5-3b__v2            all                    0.770  
                          en                     0.808  
                          fr                     0.732  
llama3.2-3b__v1           all                    0.794  
                          en                     0.820  
                          fr                     0.768

Exact Match is 0: the answers are cited paragraphs, not short extracts. On the French answers
the lexical scores measure the language gap with the English references; the multilingual
similarity does not.

## 7. RAGAS metrics that read the passages

The recorded answers keep no passages, so the served setting (qwen2.5:3b + prompt v1, corpus v2)
was generated again with its sources (`benchmark_generation --corpus v2`), then judged by
phi4-mini (`python -m src.evaluation.judge_check ragas`, `results/ragas_context.json`). Noise
sensitivity (lower is better) runs on a subset: it is the slowest metric.

In [8]:
context = json.loads(
    (ROOT / "src" / "evaluation" / "results" / "ragas_context.json").read_text(encoding="utf-8")
)
print(
    f"{context['answered']} answered of {context['questions']}; "
    f"{context['identical_to_recorded']} identical to the recorded answers; "
    f"judge {context['judge']}"
)
fmt(pd.DataFrame(context["metrics"]).T, n=0, seconds=0, seconds_per_answer=1)

94 answered of 96; 17 identical to the recorded answers; judge phi4-mini-judge


,mean,n,seconds,seconds_per_answer
faithfulness,0.947,84,2259,26.9
context_utilization,0.951,94,765,8.1
noise_sensitivity,0.176,17,1748,102.8


## 8. Is the judge right? Cohen's kappa against human labels

30 of these answers were labelled by hand without seeing the judge's verdict (1 = every claim is in the passages,
0 = at least one is not). The sheet aims at half judged faithful by RAGAS
(faithfulness = 1) and half not, but only 8 answers judged not faithful cite passages the judge
read, so it holds 22 judged faithful and 8 not. Agreement alone can come from chance; kappa
removes it.

In [9]:
kappa = json.loads(
    (ROOT / "src" / "evaluation" / "results" / "judge_kappa.json").read_text(encoding="utf-8")
)
display(
    pd.DataFrame(kappa["confusion"], index=["human 1", "human 0"], columns=["judge 1", "judge 0"])
)
print(
    f"items {kappa['items']}  observed agreement {kappa['observed_agreement']:.3f}  "
    f"chance agreement {kappa['expected_agreement']:.3f}  kappa {kappa['kappa']:.3f} "
    f"({kappa['interpretation']}, Landis & Koch)"
)

,judge 1,judge 0
human 1,14,2
human 0,8,6


items 30  observed agreement 0.667  chance agreement 0.516  kappa 0.312 (fair, Landis & Koch)


Kappa 0.312 is "fair": the judge and the labels agree on 20 of 30 answers, little more than
chance would give (0.516). 8 of the 10 disagreements are answers the judge found faithful and
the label did not: phi4-mini is more often too lenient than too strict, so the faithfulness
means of this notebook are likely optimistic. 30 answers labelled by one person keep the
kappa itself uncertain ([results.md](../../docs/module2/results.md#the-judge-against-human-labels)).